In [0]:
from pyspark.sql import functions as F

conc = spark.table("workspace.enigh.bronce_concentrado")
gast = spark.table("workspace.enigh.bronce_gastos")

print("COLUMNAS DEL CONCENTRADO:")
print(conc.columns)

def vacios(df, cols):
    fila = df.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c) for c in cols]).collect()[0]
    return fila.asDict()

print("\nDUPLICADOS")
print("Concentrado (folioviv+foliohog):", conc.count() - conc.dropDuplicates(["folioviv", "foliohog"]).count())
print("Gastos (filas idénticas):", gast.count() - gast.drop("_fecha_ingesta", "_archivo_origen").dropDuplicates().count())

print("\nNULOS O VACÍOS")
print("Concentrado:", vacios(conc, ["folioviv", "foliohog", "ing_cor", "tam_loc", "est_socio", "sexo_jefe", "tot_integ", "factor"]))
print("Gastos:", vacios(gast, ["folioviv", "foliohog", "clave", "gasto_tri", "factor"]))

print("\nGASTO_TRI")
print("No numéricos:", gast.filter(F.col("gasto_tri").isNotNull() & F.col("gasto_tri").cast("double").isNull()).count())
g = gast.withColumn("gasto_tri", F.col("gasto_tri").cast("double"))
print("Negativos:", g.filter(F.col("gasto_tri") < 0).count(), "| Ceros:", g.filter(F.col("gasto_tri") == 0).count())
print("Cuantiles 50%, 99%, 99.9%:", g.approxQuantile("gasto_tri", [0.5, 0.99, 0.999], 0.001), "| Máximo:", g.agg(F.max("gasto_tri")).collect()[0][0])

print("\nING_COR")
c = conc.withColumn("ing_cor", F.col("ing_cor").cast("double"))
print("Menores o iguales a 0:", c.filter(F.col("ing_cor") <= 0).count())
print("Cuantiles 50%, 80%, 99%:", c.approxQuantile("ing_cor", [0.5, 0.8, 0.99], 0.001), "| Máximo:", c.agg(F.max("ing_cor")).collect()[0][0])

COLUMNAS DEL CONCENTRADO:
['folioviv', 'foliohog', 'ubica_geo', 'tam_loc', 'est_socio', 'est_dis', 'upm', 'factor', 'clase_hog', 'sexo_jefe', 'edad_jefe', 'educa_jefe', 'tot_integ', 'hombres', 'mujeres', 'mayores', 'menores', 'p12_64', 'p65mas', 'ocupados', 'percep_ing', 'perc_ocupa', 'ing_cor', 'ingtrab', 'trabajo', 'sueldos', 'horas_extr', 'comisiones', 'aguinaldo', 'indemtrab', 'otra_rem', 'remu_espec', 'negocio', 'noagrop', 'industria', 'comercio', 'servicios', 'agrope', 'agricolas', 'pecuarios', 'reproducc', 'pesca', 'otros_trab', 'rentas', 'utilidad', 'arrenda', 'transfer', 'jubilacion', 'becas', 'donativos', 'remesas', 'bene_gob', 'transf_hog', 'trans_inst', 'estim_alqu', 'otros_ing', 'gasto_mon', 'alimentos', 'ali_dentro', 'cereales', 'carnes', 'pescado', 'leche', 'huevo', 'aceites', 'tuberculo', 'verduras', 'frutas', 'azucar', 'cafe', 'especias', 'otros_alim', 'bebidas', 'ali_fuera', 'tabaco', 'vesti_calz', 'vestido', 'calzado', 'vivienda', 'alquiler', 'pred_cons', 'agua', 'en

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8435418227051460>, line 24
     22 print("No numéricos:", gast.filter(F.col("gasto_tri").isNotNull() & F.col("gasto_tri").cast("double").isNull()).count())
     23 g = gast.withColumn("gasto_tri", F.col("gasto_tri").cast("double"))
---> 24 print("Negativos:", g.filter(F.col("gasto_tri") < 0).count(), "| Ceros:", g.filter(F.col("gasto_tri") == 0).count())
     25 print("Cuantiles 50%, 99%, 99.9%:", g.approxQuantile("gasto_tri", [0.5, 0.99, 0.999], 0.001), "| Máximo:", g.agg(F.max("gasto_tri")).collect()[0][0])
     27 print("\nING_COR")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:381, in DataFrame.count(self)
    380 def count(self) -> int:
--> 381     table, _ = self.agg(F._invoke_function("count", F.lit(1)))._to_table()  # type: ignore[operator]
    382     return table[0][0].as_

In [0]:
from pyspark.sql import functions as F

gast = spark.table("workspace.enigh.bronce_gastos")
conc = spark.table("workspace.enigh.bronce_concentrado")

def num(c):
    return F.expr(f"try_cast(trim({c}) as double)")

g = gast.withColumn("gasto_tri_n", num("gasto_tri")).withColumn("gas_nm_tri_n", num("gas_nm_tri"))

print("GASTO_TRI")
print("Vacío o no numérico:", g.filter(F.col("gasto_tri_n").isNull()).count())
print("  de esos, con gas_nm_tri con valor:", g.filter(F.col("gasto_tri_n").isNull() & F.col("gas_nm_tri_n").isNotNull()).count())
print("Negativos:", g.filter(F.col("gasto_tri_n") < 0).count(), "| Ceros:", g.filter(F.col("gasto_tri_n") == 0).count())
print("Cuantiles 50%, 99%, 99.9%:", g.approxQuantile("gasto_tri_n", [0.5, 0.99, 0.999], 0.001), "| Máximo:", g.agg(F.max("gasto_tri_n")).collect()[0][0])

print("\nING_COR")
c = conc.withColumn("ing_cor_n", num("ing_cor"))
print("No numéricos:", c.filter(F.col("ing_cor_n").isNull()).count())
print("Menores o iguales a 0:", c.filter(F.col("ing_cor_n") <= 0).count())
print("Cuantiles 50%, 80%, 99%:", c.approxQuantile("ing_cor_n", [0.5, 0.8, 0.99], 0.001), "| Máximo:", c.agg(F.max("ing_cor_n")).collect()[0][0])

GASTO_TRI
Vacío o no numérico: 425861
  de esos, con gas_nm_tri con valor: 421365
Negativos: 0 | Ceros: 1805
Cuantiles 50%, 99%, 99.9%: [347.14, 7258.06, 1164130.43] | Máximo: 1164130.43

ING_COR
No numéricos: 0
Menores o iguales a 0: 6
Cuantiles 50%, 80%, 99%: [55666.18, 99719.48, 302844.7] | Máximo: 17431977.54
